In [ ]:
# Scratchpad
- Fix Daily Tasks UI
- Fix Daily Tasks 500 error
- Fix Chat List loading error

In [ ]:
# Fix 401 in Daily Tasks
The `checkSessionActive` middleware does not set `req.user`. It only checks session validity.
Controllers like `chat.ts` manually decode the token.
We need to update `dailyTasks.ts` to:
1. Import `verifyToken` from `./auth`
2. Extract token from header `req.headers.authorization`
3. Decode user ID using `verifyToken`
4. Use this ID for queries.

Files to edit:
- `backend/src/dailyTasks.ts`


In [ ]:
# Update dailyTasks.ts
1. Add import: `import { verifyToken } from './auth';`
2. Define helper function `getUserIdFromRequest(req)` to DRY.
3. Update `getDailyTask`, `openDailyTask`, `completeDailyTask` to use helper.

```typescript
import { verifyToken } from './auth';

const getUserIdFromRequest = (req: Request): number => {
    const token = req.headers.authorization?.replace('Bearer ', '');
    if (!token) throw new Error('No token provided');
    const decoded = verifyToken(token);
    if (!decoded || !decoded.userId) throw new Error('Invalid token');
    return decoded.userId;
}
```

Then in handlers:
```typescript
try {
    const userId = getUserIdFromRequest(req);
    // ...
} catch (e: any) {
    if (e.message.includes('token') || e.message.includes('jwt')) {
         return res.status(401).json({ success: false, error: 'Unauthorized' });
    }
    // ...
}
```
No, better to just copy the block or use a safe helper. The current error handling in handlers is loose.
I will write the code inline for safer edits.


In [ ]:
# Remove Voice from ChatPage.tsx

1. **Imports**:
    - Remove `AudioPlayer`, `VoiceMessageDisplay`, `VoiceAssistant`, `VoiceRecordingInput`.
    - Remove `voiceApi` from `api`.
    - Remove hooks `useVoiceMessages`, `useVoiceMessagesWithLanguage`.
    - Remove `SupportedLanguage`.

2. **State & Hooks**:
    - Remove `voiceMessages` definition.
    - Remove `showVoiceAssistant`.
    - Remove `selectedLanguage`.

3. **Handlers**:
    - Remove `handleVoiceRecordingStart`.
    - Remove `handleTranscribeVoice`.
    - Remove `handleSendVoiceMessage`.
    - Remove `handleVoiceAssistantClose` (if exists).

4. **Render**:
    - Remove `VoiceAssistant` component.
    - Modify `renderMessage` to remove `VoiceMessageDisplay` check.
    - Replace `VoiceRecordingInput` with `TextInput` + `TouchableOpacity` (Send).
    - Remove `voiceMessages.error` banner.

5. **Interface**:
    - Remove `audioUri`, `isVoiceMessage`, `transcript` etc from `Message` interface (or just ignore them).

6. **Cleanup**:
    - Delete unused files later: `VoiceRecordingInput.tsx`, `VoiceMessageDisplay.tsx`, `VoiceAssistant.tsx`, `AudioPlayer.tsx`, `useVoiceMessages.ts`, `VoiceService.ts`.


In [ ]:
# Remove Voice from api.ts

1. **Interfaces**:
    - `ChatRequest`: Remove `audioUri`, `isVoiceMessage`, `language`, `transcript`, `isTranscribed`.
    - `ChatHistoryMessage`: Remove `audio_uri`, `language`, `transcript`, `is_transcribed`, `tts_audio_uri`, `is_voice_response`.
    - definition of `VoiceMessageRequest`, `VoiceMessageResponse`.

2. **API Object**:
    - Remove `voiceApi` object entirely.

3. **Functions**:
    - Update `sendMessage` in `chatApi` to remove voice fields from payload.


In [ ]:
# Fix backend/src/chat.ts

For `createChat`, `getChatList`, `getChatMessages`, `saveChatMessage`, `deleteChat`:

```typescript
    const token = req.headers.authorization?.replace('Bearer ', '');
    if (!token) {
        return res.status(401).json({ success: false, error: 'Unauthorized' });
    }

    let userId: number;
    try {
        const decoded = verifyToken(token);
        userId = decoded.userId;
    } catch (e) {
        return res.status(401).json({ success: false, error: 'Unauthorized: Invalid token' });
    }
```

For `getChatMessages`:
Remove `audio_uri`, `language`, `transcript`, `is_transcribed`, `tts_audio_uri`, `is_voice_response` from SQL.

For `saveChatMessage`:
Remove saving `mood`? No, keep mood. But ensure `sender` logic is clean.


In [ ]:
# Update tasks.tsx UI

- Import `Animated, { FadeInUp, FadeInDown, ZoomIn, Layout }` from `react-native-reanimated`.
- Calculate `timeLeft` string (HH:MM:SS) to midnight.
- Add `isDark` check: `const isDark = activeTheme === 'dark';`
- Update gradients:
    - Background: Dark: `['#1E293B', '#0F172A']`, Light: `['#F1F5F9', '#E2E8F0']`
    - Card: Use more vibrant gradients.
- Add Timer display in header or footer.
- Add animations to the card and button.
